## Assignment 6 - customer segementation

In [105]:
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score

In [106]:

CSV_PATH = "dataset/raw_wholesale_customers.csv"
OUT_PATH = "dataset/last_segmented_wholesale_customers.csv"


In [107]:
## constants 
FEATURES = ["Fresh", "Milk", "Grocery", "Frozen", "Detergents_Paper", "Delicassen"]

RANDOM_STATE = 42
K = 5

### Load The Dataset

In [108]:
### Load the dataset
df = pd.read_csv(CSV_PATH)

### Initial Data

In [109]:
print(df.head())

   Channel  Region  Fresh  Milk  Grocery  Frozen  Detergents_Paper  Delicassen
0        2       3  12669  9656     7561     214              2674        1338
1        2       3   7057  9810     9568    1762              3293        1776
2        2       3   6353  8808     7684    2405              3516        7844
3        1       3  13265  1196     4221    6404               507        1788
4        2       3  22615  5410     7198    3915              1777        5185


### Feature X

In [110]:
x = df[FEATURES].copy()

print("\n=== FEATURE MATRIX ===\n======================\n")
print(x.head())


=== FEATURE MATRIX ===

   Fresh  Milk  Grocery  Frozen  Detergents_Paper  Delicassen
0  12669  9656     7561     214              2674        1338
1   7057  9810     9568    1762              3293        1776
2   6353  8808     7684    2405              3516        7844
3  13265  1196     4221    6404               507        1788
4  22615  5410     7198    3915              1777        5185


In [111]:
def outliersIQR(dataframe, column_name, multiplier = 1.5 ):

    # 1. Calculate Q1, Q3, and IQR
    q1 = dataframe[column_name].quantile(0.25)
    q3 = dataframe[column_name].quantile(0.75)
    iqr = q3 - q1

    # 2. Calculate upper and lower fences
    lower_limit = q1 - (multiplier * iqr)
    upper_limit = q3 + (multiplier * iqr)

    return lower_limit, upper_limit

# Example usage:
# lower, upper = outliersIQR(x, 'Milk', multiplier=1.5)

low_fresh, high_fresh = outliersIQR(x, 'Fresh')
low_milk, high_milk = outliersIQR(x, 'Milk')
low_grocery, high_grocery = outliersIQR(x, 'Grocery')
low_frozen, high_frozen = outliersIQR(x, 'Frozen')
low_detergents, high_detergents = outliersIQR(x, 'Detergents_Paper')
low_delicassen, high_delicassen = outliersIQR(x, 'Delicassen')


# clip the outliers
x['Fresh'] = x['Fresh'].clip(lower=low_fresh, upper=high_fresh)
x['Milk'] = x['Milk'].clip(lower=low_milk, upper=high_milk)
x['Grocery'] = x['Grocery'].clip(lower=low_grocery, upper=high_grocery)
x['Frozen'] = x['Frozen'].clip(lower=low_frozen, upper=high_frozen)
x['Detergents_Paper'] = x['Detergents_Paper'].clip(lower=low_detergents, upper=high_detergents)
x['Delicassen'] = x['Delicassen'].clip(lower=low_delicassen, upper=high_delicassen)


df[FEATURES] = x


## IQR CAP SUMMARY

In [112]:
print("\n=== OUTLIER CAPPING ===\n=======================\n")
print(f"Fresh low={low_fresh:.2f}  high={high_fresh:.2f}  |  after cap min={x['Fresh'].min():.2f}  max={x['Fresh'].max():.2f}")
print(f"Milk low={low_milk:.2f}  high={high_milk:.2f}  |  after cap min={x['Milk'].min():.2f}  max={x['Milk'].max():.2f}")
print(f"Grocery low={low_grocery:.2f}  high={high_grocery:.2f}  |  after cap min={x['Grocery'].min():.2f}  max={x['Grocery'].max():.2f}")
print(f"Frozen low={low_frozen:.2f}  high={high_frozen:.2f}  |  after cap min={x['Frozen'].min():.2f}  max={x['Frozen'].max():.2f}")
print(f"Detergents_Paper low={low_detergents:.2f}  high={high_detergents:.2f}  |  after cap min={x['Detergents_Paper'].min():.2f}  max={x['Detergents_Paper'].max():.2f}")
print(f"Delicassen low={low_delicassen:.2f}  high={high_delicassen:.2f}  |  after cap min={x['Delicassen'].min():.2f}  max={x['Delicassen'].max():.2f}")



=== OUTLIER CAPPING ===

Fresh low=-17581.25  high=37642.75  |  after cap min=3.00  max=37642.75
Milk low=-6952.88  high=15676.12  |  after cap min=55.00  max=15676.12
Grocery low=-10601.12  high=23409.88  |  after cap min=3.00  max=23409.88
Frozen low=-3475.75  high=7772.25  |  after cap min=25.00  max=7772.25
Detergents_Paper low=-5241.12  high=9419.88  |  after cap min=3.00  max=9419.88
Delicassen low=-1709.75  high=3938.25  |  after cap min=3.00  max=3938.25


### Head after IQR CAP

In [113]:
print("\n=== FEATURES HEAD (after IQR cap) ===\n=====================================\n")
print(x.head())


=== FEATURES HEAD (after IQR cap) ===

     Fresh    Milk  Grocery  Frozen  Detergents_Paper  Delicassen
0  12669.0  9656.0   7561.0   214.0            2674.0     1338.00
1   7057.0  9810.0   9568.0  1762.0            3293.0     1776.00
2   6353.0  8808.0   7684.0  2405.0            3516.0     3938.25
3  13265.0  1196.0   4221.0  6404.0             507.0     1788.00
4  22615.0  5410.0   7198.0  3915.0            1777.0     3938.25


## StandarScaler

In [114]:
scaler = StandardScaler()
x_scaler = scaler.fit_transform(x)

print("\n === Scaled shape ===\n=====================\n", x_scaler.shape)


 === Scaled shape ===
 (440, 6)


### Elbow Method

In [115]:
print("\n=== ELBOW METHOD (SSE per k) ===\n=====================================\n")
for k in range(1, 11):
    km = KMeans(n_clusters=k, n_init="auto", random_state=RANDOM_STATE)
    km.fit(x_scaler)
    print(f"k={k} → SSE={km.inertia_:.2f}")


=== ELBOW METHOD (SSE per k) ===



k=1 → SSE=2640.00
k=2 → SSE=1728.19
k=3 → SSE=1363.45
k=4 → SSE=1202.41
k=5 → SSE=1070.15
k=6 → SSE=964.76
k=7 → SSE=921.14
k=8 → SSE=776.63
k=9 → SSE=726.88
k=10 → SSE=707.41


### Train KMeans

In [116]:
kmeans = KMeans(n_clusters=K, n_init="auto", random_state=RANDOM_STATE)
labels = kmeans.fit_predict(x_scaler)

df["Cluster"] = labels.astype(int)
print("\n=== SAMPLE WITH CLUSTERS ===\n============================\n")
df.head()


=== SAMPLE WITH CLUSTERS ===



,Channel,Region,Fresh,Milk,Grocery,Frozen,Detergents_Paper,Delicassen,Cluster
0,2,3,12669.0,9656.0,7561.0,214.0,2674.0,1338.00,0
1,2,3,7057.0,9810.0,9568.0,1762.0,3293.0,1776.00,0
2,2,3,6353.0,8808.0,7684.0,2405.0,3516.0,3938.25,0
3,1,3,13265.0,1196.0,4221.0,6404.0,507.0,1788.00,3
4,2,3,22615.0,5410.0,7198.0,3915.0,1777.0,3938.25,3


### Evaluate clustring

In [117]:
sil = silhouette_score(x_scaler, labels)
dbi = davies_bouldin_score(x_scaler, labels)
print("\n=== METRICS ===\n===============\n")
print(f"Silhouette Score : {sil:.3f} (closer to +1 is better)")
print(f"Davies–Bouldin   : {dbi:.3f} (lower is better)")


=== METRICS ===

Silhouette Score : 0.283 (closer to +1 is better)
Davies–Bouldin   : 1.270 (lower is better)


### Cluster centers

In [118]:
centers_scaled = kmeans.cluster_centers_
centers_original = scaler.inverse_transform(centers_scaled)

centers_df = pd.DataFrame(centers_original, columns=FEATURES)
centers_df.index.name = "Cluster"

print("\n=== CLUSTER CENTERS (Original Units) ===\n========================================\n")
centers_df.round(2)


=== CLUSTER CENTERS (Original Units) ===



,Fresh,Milk,Grocery,Frozen,Detergents_Paper,Delicassen
Cluster,,,,,,
0,9202.67,6833.30,9104.12,1326.16,3280.12,1871.76
1,8376.23,2150.65,3160.63,1646.33,779.25,674.02
2,17461.54,13805.60,17524.12,4120.57,5460.56,3583.64
3,22346.70,3409.14,3969.33,5819.60,583.07,1566.95
4,4916.98,10768.85,18350.13,1212.37,7780.02,981.37


# ! USING DBSCAN AS SECOND CLUSTERING

- Handles Varied Buying Densities: Wholesale customers naturally form highly dense clusters (like small independent grocery stores with standard order sizes) alongside sparse, high-volume clusters (like massive chain distributors). DBSCAN groups them by density rather than forcing them into uniform spherical clusters like K-Means.
- Built-in Outlier Identification: The dataset contains extreme purchasing volumes (wholesalers ordering massive amounts of Fresh or Grocery goods). DBSCAN automatically flags these unique, high-value corporate clients as noise (-1) instead of pulling the cluster centers away from the typical customer profiles.
- No Pre-set Cluster Number (K): We do not have to guess or arbitrarily choose how many customer segments exist via an elbow method; the algorithm naturally discovers the true number of clusters based on purchasing patterns.

https://scikit-learn.org/stable/modules/generated/sklearn.cluster.DBSCAN.html

In [119]:
# === 1. Train DBSCAN ===
# eps: The maximum distance between two samples for one to be considered as in the neighborhood of the other.
# min_samples: The number of samples in a neighborhood for a point to be considered as a core point.
EPS_VAL = 1.0
MIN_SAMPLES_VAL = 5

dbscan = DBSCAN(eps=EPS_VAL, min_samples=MIN_SAMPLES_VAL)
labels = dbscan.fit_predict(x_scaler)


# Add labels to dataframe
df["DBSCAN"] = labels.astype(int)

print("\n=== SAMPLE WITH CLUSTERS ===\n============================\n")
df.head()



=== SAMPLE WITH CLUSTERS ===



,Channel,Region,Fresh,Milk,Grocery,Frozen,Detergents_Paper,Delicassen,Cluster,DBSCAN
0,2,3,12669.0,9656.0,7561.0,214.0,2674.0,1338.00,0,0
1,2,3,7057.0,9810.0,9568.0,1762.0,3293.0,1776.00,0,0
2,2,3,6353.0,8808.0,7684.0,2405.0,3516.0,3938.25,0,-1
3,1,3,13265.0,1196.0,4221.0,6404.0,507.0,1788.00,3,0
4,2,3,22615.0,5410.0,7198.0,3915.0,1777.0,3938.25,3,3


In [120]:

# === 2. Evaluate Clustering ===
# DBSCAN targets noise/outliers as -1. 
# Silhouette score metrics fail if only 1 cluster (+ noise) is found, so we check first.
unique_clusters = set(labels) - {-1}
num_clusters = len(unique_clusters)

print("\n=== METRICS ===\n===============\n")
print(f"Total Clusters Found (excluding Noise): {num_clusters}")
print(f"Noise Points Detected (-1): {list(labels).count(-1)}")


=== METRICS ===

Total Clusters Found (excluding Noise): 5
Noise Points Detected (-1): 99


In [121]:
if num_clusters > 1:
    sil = silhouette_score(x_scaler, labels)
    dbi = davies_bouldin_score(x_scaler, labels)
    print(f"Silhouette Score : {sil:.3f} (closer to +1 is better)")
    print(f"Davies_Bouldin   : {dbi:.3f} (lower is better)")
else:
    print("Metrics skipped: Not enough distinct clusters found to calculate silhouette/DB metrics.")

Silhouette Score : 0.057 (closer to +1 is better)
Davies_Bouldin   : 1.507 (lower is better)


In [122]:
# === 3. Cluster Centers (Group Means) ===
print("\n=== CLUSTER CENTERS (Original Units Group Means) ===\n====================================================\n")
# Since DBSCAN doesn't have a native '.cluster_centers_' attribute like KMeans, 
# we calculate the mean profile of the original units grouped by cluster label.
cluster_profiles = df.groupby("Cluster")[FEATURES].mean()
print(cluster_profiles.round(2))


=== CLUSTER CENTERS (Original Units Group Means) ===

            Fresh      Milk   Grocery   Frozen  Detergents_Paper  Delicassen
Cluster                                                                     
0         9202.67   6833.30   9104.12  1326.16           3280.12     1871.76
1         8376.23   2150.65   3160.63  1646.33            779.25      674.02
2        17461.54  13805.60  17524.12  4120.57           5460.56     3583.64
3        22346.70   3409.14   3969.33  5819.60            583.07     1566.95
4         4916.98  10768.85  18350.13  1212.37           7780.02      981.37


### Sanity Check

In [123]:
sample_idx = [0, 1, 2, 3]
sanity = df.loc[sample_idx, ["Channel", "Region"] + FEATURES + ["Cluster"]]
print("\n=== SANITY CHECK (4 Clients) ===\n================================\n")
sanity


=== SANITY CHECK (4 Clients) ===



,Channel,Region,Fresh,Milk,Grocery,Frozen,Detergents_Paper,Delicassen,Cluster
0,2,3,12669.0,9656.0,7561.0,214.0,2674.0,1338.00,0
1,2,3,7057.0,9810.0,9568.0,1762.0,3293.0,1776.00,0
2,2,3,6353.0,8808.0,7684.0,2405.0,3516.0,3938.25,0
3,1,3,13265.0,1196.0,4221.0,6404.0,507.0,1788.00,3


# SAVE LEBELED

In [124]:
df.to_csv(OUT_PATH, index=False)
print(f"\nSaved clustered dataset → {OUT_PATH}")


Saved clustered dataset → dataset/last_segmented_wholesale_customers.csv
